# 🚀 AI Designer Backend — Colab Edition v3

**Thay đổi v3:** Dùng `nohup` + `disown` thay vì `subprocess.Popen` để server không bị Colab kill.

---

### CELL 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

---

### CELL 2: Clone repo (CHẠY 1 LẦN, sau đó comment)

In [ ]:
# THAY ĐỔI URL NÀY bằng repo của bạn!
GIT_URL = 'https://github.com/YOUR_USERNAME/AI_GEN.git'  # @param {type:"string"}

!git clone $GIT_URL /content/drive/MyDrive/AI_GEN_IMAGE 2>&1 | tail -5
print('✅ Cloned')

---

### CELL 3: Cài Dependencies (1 LẦN, sau comment)

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q 2>&1 | tail -3
!pip install fastapi uvicorn[standard] python-multipart aiofiles pydantic python-dotenv -q 2>&1 | tail -1
!pip install diffusers transformers accelerate peft safetensors huggingface_hub -q 2>&1 | tail -1
!pip install opencv-python opencv-contrib-python scikit-image scipy Pillow numpy tqdm -q 2>&1 | tail -1
!pip install open-clip-torch pyngrok -q 2>&1 | tail -1
print('✅ Dependencies installed')

---

### CELL 4: Nhập tokens (MỖI LẦN RECONNECT)

In [ ]:
import os, getpass

HF_TOKEN = getpass.getpass('🔑 HuggingFace Token: ')
os.environ['HF_TOKEN'] = HF_TOKEN

NGROK_TOKEN = getpass.getpass('🔑 ngrok Authtoken: ')
os.environ['NGROK_TOKEN'] = NGROK_TOKEN

from pyngrok import conf
conf.get_default().auth_token = NGROK_TOKEN

print('✅ Tokens OK')

---

### CELL 5: Setup project + Fix CORS (1 LẦN)

In [ ]:
# CELL FIX CORS — CHẠY TRỰC TIẾP TRÊN COLAB
import os

PROJECT_DIR = '/content/drive/MyDrive/AI_GEN_IMAGE'
main_py = f'{PROJECT_DIR}/app/main.py'

if not os.path.exists(main_py):
    print(f'❌ File not found: {main_py}')
else:
    with open(main_py, 'r') as f:
        content = f.read()
    
    # Thay thế đoạn CORS cũ bằng đoạn mới
    old_cors = '''allow_origins=["http://localhost:5173", "http://localhost:3000"],
    allow_credentials=True,'''
    
    new_cors = '''allow_origins=["*"],
    allow_credentials=False,'''
    
    if old_cors in content:
        content = content.replace(old_cors, new_cors)
        with open(main_py, 'w') as f:
            f.write(content)
        print('✅ CORS fixed! allow_origins=["*"]')
    else:
        print('⚠️ Không tìm thấy CORS cũ — file có thể đã được fix')
        print('  Kiểm tra file:')
        lines = [l for l in content.split('\n') if 'allow_origins' in l or 'allow_credentials' in l]
        for l in lines:
            print(f'  {l}')

---

### CELL 6: Khởi động Backend ⭐ (GIỮ CHẠY MÃI!)

**LƯU Ý:** Cell này sẽ in ra ngrok URL. COPY URL đó. ĐỪNG STOP cell này.

In [ ]:
# ⭐ CELL QUAN TRỌNG NHẤT

import os
import subprocess
import time
import urllib.request
import json
from pyngrok import ngrok

PROJECT_DIR = '/content/drive/MyDrive/AI_GEN_IMAGE'
os.chdir(PROJECT_DIR)
os.environ['PYTHONPATH'] = PROJECT_DIR

# Kill cũ
subprocess.run(['bash', '-c', 'pkill -f uvicorn 2>/dev/null; pkill -f ngrok 2>/dev/null; sleep 1'], capture_output=True)

# 1. Mở ngrok tunnel
print('🔓 Opening ngrok tunnel...')
try:
    ngrok.kill()
except:
    pass
time.sleep(1)

http_tunnel = ngrok.connect(8000, bind_tls=True)
public_url = http_tunnel.public_url
api_url = public_url + '/api'

print()
print('=' * 65)
print('🎉 BACKEND PUBLIC URL:')
print(f'   {public_url}')
print(f'   API Base: {api_url}')
print('=' * 65)
print()
# 2. Start backend với nohup (không bị kill khi cell end)
print('🚀 Starting FastAPI backend...')
print('   (Lần đầu: tải SDXL Turbo ~6GB — đợi 3-10 phút)')
print()
log_path = f'{PROJECT_DIR}/backend.log'

# Dùng nohup để process không bị kill khi cell kết thúc
cmd = [
    'bash', '-c',
    f'cd {PROJECT_DIR} && '
    f'PYTHONPATH={PROJECT_DIR} '
    f'HF_TOKEN={os.environ.get("HF_TOKEN", "")} '
    f'nohup python -m uvicorn app.main:app '
    f'--host 0.0.0.0 --port 8000 '
    f'--log-level info > {log_path} 2>&1 &'
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(f'   Command: {" ".join(cmd[:3])} ...')
print(f'   Log: {log_path}')
print()

# 3. Đợi server khởi động
print('⏳ Đang đợi server (xem log bên dưới)...')
print()

# Đợi và kiểm tra health
server_ready = False
for attempt in range(20):
    time.sleep(10)
    try:
        resp = urllib.request.urlopen(f'{public_url}/health', timeout=8)
        data = json.loads(resp.read())
        print('✅' * 30)
        print('🎉 SERVER SẴN SÀNG!')
        print(f'   Backend: {data.get("backend")}')
        print(f'   GPU: {data.get("gpu_available")}')
        print(f'   URL: {public_url}')
        print('✅' * 30)
        server_ready = True
        break
    except Exception as e:
        err_str = str(e)[:60]
        print(f'   Thử {attempt+1}/20: {err_str}...')

if not server_ready:
    print()
    print('⚠️  Server chưa reply sau 200s.')
    print('   Xem log: !cat /content/drive/MyDrive/AI_GEN_IMAGE/backend.log')
    print()

# 4. In log tail
print()
print('📋 Backend log (cuối 40 dòng):')
print('-' * 60)
try:
    with open(log_path, 'r') as f:
        lines = f.readlines()
    for line in lines[-40:]:
        print(line.rstrip())
except:
    print('(log chưa có nội dung)')
print('-' * 60)

---

### CELL 7: In log real-time (CHẠY SONG SONG, cell 6 vẫn chạy)

In [ ]:
# Chạy cell này để xem log liên tục
# Cell 6 vẫn tiếp tục chạy bên dưới!

!cat /content/drive/MyDrive/AI_GEN_IMAGE/backend.log | tail -60

---

### CELL 8: Kiểm tra server

In [ ]:
# Test các endpoint
import urllib.request, json, re, os

PROJECT_DIR = '/content/drive/MyDrive/AI_GEN_IMAGE'
log_path = f'{PROJECT_DIR}/backend.log'

# Tìm ngrok URL trong log
public_url = None
try:
    with open(log_path, 'r') as f:
        log = f.read()
    match = re.search(r'https://[^\s]+\.ngrok-free\.app', log)
    if match:
        public_url = match.group()
except:
    pass

if not public_url:
    print('❌ Chưa có ngrok URL — chạy cell 6 trước')
else:
    api_url = public_url + '/api'
    print(f'🔗 Backend: {public_url}')
    print()

    for ep, label in [
        ('/health', 'Health'),
        ('/api/model/status', 'Model Status'),
        ('/api/lora', 'LoRA'),
        ('/api/export/formats', 'Export Formats'),
    ]:
        try:
            r = urllib.request.urlopen(f'{public_url}{ep}', timeout=10)
            d = json.loads(r.read())
            print(f'✅ {label}: OK')
        except Exception as e:
            print(f'❌ {label}: {str(e)[:50]}')

    print()
    print('📋 Log tail:')
    !tail -20 /content/drive/MyDrive/AI_GEN_IMAGE/backend.log

---

## 🔧 Nếu server không chạy — Debug

```bash
# 1. Xem toàn bộ log
!cat /content/drive/MyDrive/AI_GEN_IMAGE/backend.log

# 2. Kiểm tra uvicorn process
!ps aux | grep uvicorn

# 3. Thử chạy thủ công để xem lỗi
# cd /content/drive/MyDrive/AI_GEN_IMAGE
# python -m uvicorn app.main:app --host 0.0.0.0 --port 8000
```